In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import deeplake as dl
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

In [ ]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
test_ds = dl.load("hub://activeloop/visdrone-det-test-dev")

In [ ]:
# Global Variables and Configs
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Image Constants
IMG_HEIGHT = 224
IMG_WIDTH = 407
IMG_CHANNELS = 3
IMAGE_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

GLOBAL_BACKBONE = keras.applications.MobileNetV2(
    weights='imagenet', 
    include_top=False, 
    pooling='avg', 
    input_shape=IMAGE_SHAPE
)
GLOBAL_BACKBONE.trainable = False

city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

In [4]:
# Preprocessing Detection

def image_preprocessing(image):
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def detection_preprocessing(data):
    image = image_preprocessing(data['images'])
    labels = data['labels']  # Array of integer Class IDs
    
    # 1. Track Humans: Pedestrians (1) OR People (2)
    is_human = tf.reduce_any(tf.equal(tf.expand_dims(labels, -1), [1, 2]), axis=-1)
    human_count = tf.reduce_sum(tf.cast(is_human, tf.float32))
    
    # 2. Track Vehicles: Car (4), Vans (5), Trucks (6), Buses (9), Motorcycles (10)
    is_vehicle = tf.reduce_any(tf.equal(tf.expand_dims(labels, -1), [4, 5, 6, 9, 10]), axis=-1)
    vehicle_count = tf.reduce_sum(tf.cast(is_vehicle, tf.float32))
    
    counts_vector = tf.stack([human_count, vehicle_count], axis=0)
    return image, counts_vector

# Feature Stream Pipelines

train_stream = (train_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .cache()
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

val_stream = (val_ds.tensorflow()
                .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                .cache()
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

test_stream = (test_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

In [6]:
# Preprocessing City Labels
@tf.function
def forward_pass(images):
    return GLOBAL_BACKBONE(images, training=False)

def extract_features(dataset_stream):
    features_list = []
    fast_stream = dataset_stream.unbatch().batch(32).prefetch(2)
    
    for img_batch, _ in fast_stream:
        batch_features = forward_pass(img_batch)
        features_list.append(batch_features.numpy())
        
    return np.concatenate(features_list, axis=0)

train_features = extract_features(train_stream)
val_features = extract_features(val_stream)
test_features = extract_features(test_stream)

trained_pca = PCA(n_components=50, random_state=42)
trained_kmeans = MiniBatchKMeans(n_clusters=14, random_state=42, n_init=5, batch_size=1024)

tf_train_labels = trained_kmeans.fit_predict(trained_pca.fit_transform(train_features))
tf_val_labels = trained_kmeans.predict(trained_pca.transform(val_features))
tf_test_labels = trained_kmeans.predict(trained_pca.transform(test_features))

train_labels_ds = tf.data.Dataset.from_tensor_slices(tf_train_labels).batch(BATCH_SIZE)
val_labels_ds = tf.data.Dataset.from_tensor_slices(tf_val_labels).batch(BATCH_SIZE)
test_labels_ds = tf.data.Dataset.from_tensor_slices(tf_test_labels).batch(BATCH_SIZE)

In [ ]:
# Adding Pipelines

def shape_structure_outputs(img_batch, count_batch, label_batch):
    # Enforce static ranks for the Keras graph builder
    img_batch.set_shape([None, *IMAGE_SHAPE])
    count_batch.set_shape([None, 2])
    label_batch.set_shape([None])
    
    return img_batch, {
        "city_output": label_batch,
        "count_output": count_batch
    }

# Zip your feature streams (images + counts) with your clustering labels
train_pipeline = tf.data.Dataset.zip((train_stream, train_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_pipeline = tf.data.Dataset.zip((val_stream, val_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_pipeline = tf.data.Dataset.zip((test_stream, test_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

In [12]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(IMAGE_SHAPE), name='input_image')

    base_model = keras.applications.MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_tensor=input_tensor
    )
    base_model.trainable = False

    shared_features = base_model.output

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.Conv2D(32, (3, 3), padding="same", activation='relu', kernel_initializer='he_normal')(shared_features)
    x_detect = layers.GlobalMaxPooling2D()(x_detect)
    
    x_detect = layers.Dense(128, activation=None, kernel_initializer='he_normal')(x_detect)
    x_detect = layers.BatchNormalization()(x_detect)
    x_detect = layers.LeakyReLU(negative_slope=0.1)(x_detect)
    
    # Final count output layer
    count_output = layers.Dense(2, activation='softplus', kernel_initializer='he_normal', name="count_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    
    protected_counts = layers.Lambda(lambda x: tf.stop_gradient(x))(count_output)

    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, protected_counts])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, count_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "count_output": "huber"
        },
        metrics={
            "city_output": "accuracy",
            "count_output": "mae"
        },
        loss_weights={
            "city_output": 0.5, 
            "count_output": 1.0
        }
    )
    return model

china_model = build_multi_head_model()

history = china_model.fit(train_pipeline, validation_data=val_pipeline, epochs=3)

print(china_model.summary())

C:\Users\etito\AppData\Local\Temp\ipykernel_16504\284182174.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


Epoch 1/3
    405/Unknown 181s 435ms/step - city_output_accuracy: 0.5395 - city_output_loss: 1.3843 - count_output_loss: 19.4506 - count_output_mae: 19.9343 - loss: 20.1427

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


405/405 ━━━━━━━━━━━━━━━━━━━━ 197s 476ms/step - city_output_accuracy: 0.6534 - city_output_loss: 0.9739 - count_output_loss: 16.2998 - count_output_mae: 16.8359 - loss: 16.8395 - val_city_output_accuracy: 0.8285 - val_city_output_loss: 0.4334 - val_count_output_loss: 11.9259 - val_count_output_mae: 12.8345 - val_loss: 12.5705
Epoch 2/3
405/405 ━━━━━━━━━━━━━━━━━━━━ 194s 479ms/step - city_output_accuracy: 0.7753 - city_output_loss: 0.5997 - count_output_loss: 11.1192 - count_output_mae: 11.6321 - loss: 11.4520 - val_city_output_accuracy: 0.8777 - val_city_output_loss: 0.3489 - val_count_output_loss: 11.4679 - val_count_output_mae: 12.3500 - val_loss: 12.0441
Epoch 3/3
405/405 ━━━━━━━━━━━━━━━━━━━━ 198s 489ms/step - city_output_accuracy: 0.8149 - city_output_loss: 0.4868 - count_output_loss: 9.1778 - count_output_mae: 9.6818 - loss: 9.4516 - val_city_output_accuracy: 0.8595 - val_city_output_loss: 0.3630 - val_count_output_loss: 11.8455 - val_count_output_mae: 12.7458 - val_loss: 12.4440


Model: "functional_3"

...

 Total params: 4,374,609 (16.69 MB)

 Trainable params: 705,456 (2.69 MB)

 Non-trainable params: 2,258,240 (8.61 MB)

 Optimizer params: 1,410,913 (5.38 MB)

None


In [13]:
# Testing Model

results = china_model.evaluate(test_pipeline, return_dict=True)

for test_images, test_targets in test_pipeline.take(1):
    city_preds, count_preds = china_model.predict(test_images, verbose=0)
    
    predicted_city_indices = np.argmax(city_preds, axis=-1)
    true_city_indices = np.array(test_targets["city_output"]).astype(int).flatten()
    true_counts = np.array(test_targets["count_output"])
    
    for i in range(min(5, len(true_city_indices))):
        pred_label = city_labels[predicted_city_indices[i]]
        true_label = city_labels[true_city_indices[i]]
        
        print(f"\nSample {i+1}:")
        print(f"  [City]  Predicted: {pred_label:<12} | True: {true_label}")
        print(f"  [Human] Predicted: {count_preds[i][0]:<12.2f} | True: {true_counts[i][0]:.1f}")
        print(f"  [Car]   Predicted: {count_preds[i][1]:<12.2f} | True: {true_counts[i][1]:.1f}")

101/101 ━━━━━━━━━━━━━━━━━━━━ 65s 638ms/step - city_output_accuracy: 0.8547 - city_output_loss: 0.3732 - count_output_loss: 13.7312 - count_output_mae: 14.3509 - loss: 14.0542

Sample 1:
  [City]  Predicted: nanyang      | True: nanyang
  [Human] Predicted: 1.95         | True: 0.0
  [Car]   Predicted: 8.74         | True: 15.0

Sample 2:
  [City]  Predicted: jinchang     | True: jinchang
  [Human] Predicted: 24.82        | True: 0.0
  [Car]   Predicted: 58.76        | True: 29.0

Sample 3:
  [City]  Predicted: jinchang     | True: jinchang
  [Human] Predicted: 28.42        | True: 54.0
  [Car]   Predicted: 52.62        | True: 48.0

Sample 4:
  [City]  Predicted: suzhou       | True: suzhou
  [Human] Predicted: 20.36        | True: 12.0
  [Car]   Predicted: 29.13        | True: 33.0

Sample 5:
  [City]  Predicted: daqing       | True: daqing
  [Human] Predicted: 7.00         | True: 0.0
  [Car]   Predicted: 30.48        | True: 37.0
